# Meta-Data Reader

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

import h5py
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

Run this cell and then enter the path to the folder in which contains all 4D-STEM scan data including HDF files containing that meta data.

In [2]:
metadata_file = '/dls/e02/data/2025/cm40603-5/processing/Merlin/C1_Au_GCN/20250813_134447/20250813_134447.hdf'

with h5py.File(metadata_file, 'r') as f:
    keys = list(f['metadata'].keys())

print(f"{'#':<3} {'Key':<35} Access line")
print("-" * 90)

for i, key in enumerate(keys, start=1):
    access = f"f['metadata']['{key}'][()]"
    print(f"{i:<3} {key:<35} '{key}': {access}")

#   Key                                 Access line
------------------------------------------------------------------------------------------
1   4D_shape                            '4D_shape': f['metadata']['4D_shape'][()]
2   A1_value_(kV)                       'A1_value_(kV)': f['metadata']['A1_value_(kV)'][()]
3   A2_value_(kV)                       'A2_value_(kV)': f['metadata']['A2_value_(kV)'][()]
4   aperture_size                       'aperture_size': f['metadata']['aperture_size'][()]
5   convergence_semi-angle(rad)         'convergence_semi-angle(rad)': f['metadata']['convergence_semi-angle(rad)'][()]
6   current_OLfine                      'current_OLfine': f['metadata']['current_OLfine'][()]
7   deflector_values                    'deflector_values': f['metadata']['deflector_values'][()]
8   defocus(nm)                         'defocus(nm)': f['metadata']['defocus(nm)'][()]
9   defocus_per_bit(nm)                 'defocus_per_bit(nm)': f['metadata']['defocus_per_bit(nm)']

In [3]:
#/dls/e02/data/2026/mg42192-4/processing/Merlin/P1

def read_metadata(file_path):
    if not os.path.exists(file_path):
        return f"Error: File does not exist at {file_path}"
    
    try:
        with h5py.File(file_path, 'r') as f:
            metadata = {
                'Step Size (Å)': f['metadata']['step_size(m)'][()] / 1e-10,
                'Convergence Semi-Angle (rad)': f['metadata']['convergence_semi-angle(rad)'][()],
                'Camera Length (m)': f['metadata']['merlin_camera_length(m)'][()],
                'High Tension Value (V)': f['metadata']['ht_value(V)'][()],
                'Defocus (nm)': f['metadata']['defocus(nm)'][()],
                'Aperture Size': f['metadata']['aperture_size'][()],
                'Current OLfine': f['metadata']['current_OLfine'][()],
                #'Field of View (m)': f['metadata']['field_of_view(m)'][()],
                'Magnification': f['metadata']['magnification'][()],
                'Nominal Camera Length (m)': f['metadata']['nominal_camera_length(m)'][()],
                #'Nominal Scan Rotation': f['metadata']['nominal_scan_rotation'][()],
                'Set Dwell Time (usec)': f['metadata']['set_dwell_time(usec)'][()],
                'Set Scan Px': f['metadata']['set_scan_px'][()],
                'Spot Size': f['metadata']['spot_size'][()],
                'z_pos(m)': f['metadata']['z_pos(m)'][()],
                'y_tilt(deg)': f['metadata']['y_tilt(deg)'][()],
                'x_tilt(deg)': f['metadata']['x_tilt(deg)'][()],
                
                        
            }
    except KeyError as e:
        return f"Metadata key not found: {e}"
    except OSError as e:
        return f"Could not open file: {e}"
    except Exception as e:
        return f"An unexpected error occurred: {e}"
    
    return metadata

def find_hdf_files_in_folders(base_dir):
    hdf_files = {}
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if os.path.isdir(folder_path):
            try:
                for item in os.listdir(folder_path):
                    file_path = os.path.join(folder_path, item)
                    if os.path.isfile(file_path) and item.lower().endswith('.hdf'):
                        hdf_files[folder] = file_path
                        break
            except Exception as e:
                print(f"Error accessing {folder_path}: {e}")
    return hdf_files

folder_path_text = widgets.Text(
    value='',
    placeholder='Enter the base directory path (e.g., /path/to/data)',
    description='Folder:',
    layout=widgets.Layout(width='80%')
)
load_button = widgets.Button(description="Scan Folders")
output_area = widgets.Output()

def on_scan_click(b):
    clear_output(wait=True)
    display(folder_path_text, load_button, output_area)

    base_directory = folder_path_text.value.strip()
    
    if not base_directory or not os.path.isdir(base_directory):
        with output_area:
            clear_output()
            print("⚠️ Please enter a valid directory path.")
        return

    hdf_files = find_hdf_files_in_folders(base_directory)

    with output_area:
        clear_output()
        if hdf_files:
            print("✅ Found .hdf files and their metadata:")
            for folder, hdf_file in hdf_files.items():
                print(f"\n📁 Folder: {folder}")
                metadata = read_metadata(hdf_file)
                if isinstance(metadata, dict):
                    for key, value in metadata.items():
                        print(f"  {key}: {value}")
                else:
                    print(metadata)
        else:
            print("❌ No .hdf files found in any subfolders.")

load_button.on_click(on_scan_click)
display(folder_path_text, load_button, output_area)

Text(value='', description='Folder:', layout=Layout(width='80%'), placeholder='Enter the base directory path (…

Button(description='Scan Folders', style=ButtonStyle())

Output()

Read M-data from a single file